# Use case 1: supplier invoice intake

Automate the front half of accounts payable for a mid-sized German beverage maker. A supplier sends a PDF invoice. The agent extracts the fields, looks them up against the ERP (DSQL), runs a three-way match against the purchase order and goods receipt, and produces a verdict: approve, flag, or hold.

## Business context

Brunnstein Beverages receives 1500-2000 supplier invoices per year. Today an AP clerk reads each PDF, matches it to a PO in SAP, checks the goods receipt, verifies the IBAN, and posts the invoice. The agent here replaces the routine ~70% of cases (clean three-way match) and escalates the rest to a human with explicit signals.

Discrepancies the agent must catch:

- IBAN fraud (a different IBAN than the supplier master) and IBAN typos
- Unknown supplier or PO
- Quantity mismatch versus goods receipt
- Unit price drift versus the PO
- Invoice arriving before the goods receipt exists

## Pipeline

```mermaid
flowchart LR
    PDF["Invoice PDF"] --> E["Extract<br/>(Claude Haiku 4.5 vision)"]
    E --> V["Validate<br/>(DSQL lookups + 3-way match)"]
    V --> D{"Decide"}
    D -->|approve| Persist[("Persist to DSQL")]
    D -->|flag / hold| Human["Human review"]
```

Each node is a function in `use_cases/automated_invoice_processing/agent/`. The graph is composed in `graph.py` and exposed via `run_on_pdf(path)`.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
project_root = here
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
import json

import pandas as pd
from IPython.display import IFrame, Markdown

from use_cases.automated_invoice_processing.agent.evaluate import evaluate_all_scenarios
from use_cases.automated_invoice_processing.agent.graph import run_on_pdf

TEST_DIR = project_root / "use_cases" / "invoice_intake" / "test_invoices"

## Test fixtures

Eight adversarial scenarios live in `test_invoices/`. Each scenario is one PDF plus a `meta.json` that declares the expected verdict and signals.

In [ ]:
fixtures = []
for d in sorted(TEST_DIR.iterdir()):
    if not d.is_dir():
        continue
    meta = json.loads(next(d.glob("*.json")).read_text())
    fixtures.append({
        "scenario": d.name,
        "expected_outcome": meta["expected_outcome"],
        "expected_signals": ", ".join(meta["expected_signals"]) or "-",
        "description": meta["description"],
    })
pd.DataFrame(fixtures)

## Walkthrough: one scenario end to end

Pick a scenario, inspect the source PDF, run the agent, compare to the expected outcome.

In [ ]:
scenario = "02_iban_fraud"
scenario_dir = TEST_DIR / scenario
meta = json.loads(next(scenario_dir.glob("*.json")).read_text())
pdf_path = next(scenario_dir.glob("*.pdf"))
Markdown(
    f"**Scenario:** {scenario}\n\n"
    f"**Description:** {meta['description']}\n\n"
    f"**Expected outcome:** `{meta['expected_outcome']}`\n\n"
    f"**Expected signals:** {meta['expected_signals']}"
)

In [ ]:
IFrame(str(pdf_path.relative_to(project_root)), width=700, height=500)

In [ ]:
verdict = run_on_pdf(str(pdf_path))
verdict.model_dump(mode="json")

In [ ]:
matched_decision = verdict.decision == meta["expected_outcome"]
expected = set(meta["expected_signals"])
actual = set(verdict.signals)
Markdown(
    f"Decision matched expected: **{matched_decision}**\n\n"
    f"Missing signals: `{sorted(expected - actual) or 'none'}`\n\n"
    f"Extra signals: `{sorted(actual - expected) or 'none'}`"
)

## Full eval over all 8 scenarios

Each row compares the agent's verdict and signals against the meta.json for that scenario.

In [ ]:
results = evaluate_all_scenarios()
df = pd.DataFrame([
    {
        "scenario": r["scenario"],
        "decision_ok": r["decision_ok"],
        "signals_ok": r["signals_ok"],
        "expected": r["expected_decision"],
        "actual": r["actual_decision"],
        "missing": ", ".join(r["missing_signals"]) or "-",
        "extra": ", ".join(r["extra_signals"]) or "-",
    }
    for r in results
])
df

In [ ]:
n_pass = int(((df['decision_ok']) & (df['signals_ok'])).sum())
print(f"{n_pass}/{len(df)} scenarios fully pass (decision + signals match)")
print(f"{int(df['decision_ok'].sum())}/{len(df)} match on decision")
print(f"{int(df['signals_ok'].sum())}/{len(df)} signals contain at least the expected codes")

## Known limitations and next steps

The current extraction step (Claude Haiku 4.5 on the PDF as a document attachment) is the dominant source of failures. The vision model occasionally drops a digit from the IBAN, which currently routes most scenarios to `flag_for_fraud_review` regardless of the real defect.

Concrete next iterations:

- Tighten the extraction prompt with explicit IBAN length constraints and a self-check pass
- Constrained structured output via a tool / JSON schema instead of free-form JSON
- Try Sonnet 4.6 for extraction while keeping Haiku 4.5 for cheaper validation steps
- MLflow tracing on every run for prompt versioning and eval score history
- Streamlit human-in-the-loop review surface for the flagged cases